## Importar librerías y definir rutas

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import chromadb
from chromadb.config import Settings
import os

# Rutas
METADATA_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/metadatos")
CHUNKS_CSV = METADATA_FOLDER / "chunks.csv"
EMBEDDINGS_NPY = METADATA_FOLDER / "embeddings.npy"
CHUNK_IDS_NPY = METADATA_FOLDER / "chunk_ids.npy"
VECTOR_STORE = Path("/home/jupyteruser/work/vector_store")

# Crear carpeta de la base vectorial
os.makedirs(VECTOR_STORE, exist_ok=True)

## Cargar datos desde archivos

In [2]:
# Cargar metadatos de chunks
df_chunks = pd.read_csv(CHUNKS_CSV)
print(f"Metadatos cargados: {df_chunks.shape[0]} chunks")

# Cargar embeddings (matriz numpy)
embeddings = np.load(EMBEDDINGS_NPY)
print(f"Embeddings cargados: {embeddings.shape}")

# Cargar chunk_ids (permitiendo pickle porque son strings)
chunk_ids = np.load(CHUNK_IDS_NPY, allow_pickle=True)
print(f"Chunk IDs cargados: {len(chunk_ids)}")

# Asegurarse de que el orden coincida
assert len(df_chunks) == embeddings.shape[0] == len(chunk_ids), "Inconsistencia en los datos"

Metadatos cargados: 3886 chunks
Embeddings cargados: (3886, 768)
Chunk IDs cargados: 3886


## Inicializar ChromaDB persistente

In [3]:
# Crear cliente ChromaDB con persistencia en disco
client = chromadb.PersistentClient(path=str(VECTOR_STORE))

# Listar colecciones existentes (por si ya hay una)
colecciones_existentes = client.list_collections()
print(f"Colecciones existentes: {colecciones_existentes}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Colecciones existentes: []


## Crear (o resetear) la colección del corpus UPeU

In [4]:
collection_name = "corpus_upeu"

# >>> Issue 4.3: capturar excepcion especifica <<<
# En chromadb 0.4.22 la excepcion al borrar una coleccion inexistente es ValueError
try:
    client.delete_collection(name=collection_name)
    print(f"Colecci\u00f3n \u0027{collection_name}\u0027 eliminada.")
except ValueError:
    print(f"No exist\u00eda colecci\u00f3n previa \u0027{collection_name}\u0027.")
except Exception as e:
    print(f"WARN al borrar colecci\u00f3n: {type(e).__name__}: {e}")

# Crear nueva coleccion
collection = client.create_collection(
    name=collection_name,
    metadata={"description": "Corpus de reglamentos universitarios UPeU", "hnsw:space": "cosine"}
)
print(f"Colecci\u00f3n \u0027{collection_name}\u0027 creada con espacio \u0027cosine\u0027.")


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


No existía colección previa 'corpus_upeu'.
Colección 'corpus_upeu' creada con espacio 'cosine'.


## Insertar documentos en lotes

In [5]:
import re

# >>> Issue 4.1: clasificación por keywords más robusta <<<
CATEGORIAS_KEYWORDS = {
    "A": [
        "estatuto", "general upeu", "defensor", "comite electoral",
        "tupa", "reglamento interno de trabajo", "honores",
    ],
    "B": [
        "estudios", "estudiante unionista", "docencia", "idiomas",
        "movilidad", "publicaciones y fondo", "pago servicios academicos",
        "becas", "admision", "credito", "matricula", "egresado",
    ],
    "C": [
        "investigacion", "investigadores", "incentivos investigacion",
        "propiedad intelectual", "codigo etica investigacion", "etica",
    ],
    "D": [
        "promocion", "recreacion", "deporte", "residencias",
        "multimedia", "seguimiento de egresados", "servicio psicologico",
    ],
    "E": [
        "politica institucional", "politica-ambiental", "ambiental",
        "capacitacion docente", "auditoria interna", "seguridad y salud",
        "comedor", "transporte", "biblioteca",
    ],
}


def obtener_categoria(doc_name):
    """Asigna categoria A-E por keywords en el nombre del PDF."""
    nombre = doc_name.lower()
    for cat, keywords in CATEGORIAS_KEYWORDS.items():
        for kw in keywords:
            if kw in nombre:
                return cat
    return "E"


# >>> Issue 4.2: extracción de artículo/sección del texto <<<
_RE_ART = re.compile(
    r'(Art[íi]culo\s+\d+[ºo°]?(?:\s*[.-]\s*\d+)?|'
    r'Cap[íi]tulo\s+[IVXLCDM\d]+|'
    r'Secci[óo]n\s+\d+|'
    r'T[íi]tulo\s+[IVXLCDM\d]+)',
    re.IGNORECASE
)


def extraer_articulo(texto):
    """Extrae el primer articulo/capitulo del texto del chunk."""
    m = _RE_ART.search(texto[:200])
    return m.group(1).strip() if m else ""


# Preparar listas para insercion (Issue 4.2: articulo y num_chars en metadata)
ids = df_chunks['chunk_id'].astype(str).tolist()
documentos = df_chunks['texto'].tolist()
metadatos = [
    {
        "documento": row['documento'],
        "categoria": obtener_categoria(row['documento']),
        "articulo": extraer_articulo(row['texto']),
        "chunk_id": str(row.get('chunk_id', idx)),
        "num_tokens": int(row.get('num_tokens', 0)),
        "num_chars": len(row['texto']),
    }
    for idx, row in df_chunks.iterrows()
]
print(f"Listas preparadas: {len(ids)} chunks, {len(metadatos)} metadatos")


Listas preparadas: 3886 chunks, 3886 metadatos


In [6]:
# Insertar por lotes de 500 chunks
BATCH_SIZE = 500
total = len(ids)
for i in range(0, total, BATCH_SIZE):
    end = min(i+BATCH_SIZE, total)
    collection.add(
        ids=ids[i:end],
        documents=documentos[i:end],
        metadatas=metadatos[i:end],
        embeddings=embeddings[i:end].tolist()  # convertir a lista para ChromaDB
    )
    print(f"Insertados chunks {i} a {end-1} de {total}")

print(f"Inserción completa. Total de documentos en colección: {collection.count()}")

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Insertados chunks 0 a 499 de 3886
Insertados chunks 500 a 999 de 3886
Insertados chunks 1000 a 1499 de 3886
Insertados chunks 1500 a 1999 de 3886
Insertados chunks 2000 a 2499 de 3886
Insertados chunks 2500 a 2999 de 3886
Insertados chunks 3000 a 3499 de 3886
Insertados chunks 3500 a 3885 de 3886
Inserción completa. Total de documentos en colección: 3886


## Verificar con una consulta de ejemplo

In [7]:
# Cargar modelo de embeddings (el mismo usado en Notebook 3) para vectorizar la consulta
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# Consulta de prueba
consulta = "¿Cuáles son los requisitos para solicitar titulación?"
print(f"Consulta: {consulta}")

# Generar embedding de la consulta
query_embedding = model.encode([consulta])[0].tolist()

# Buscar en ChromaDB
resultados = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,  # top 3 chunks más relevantes
    include=["documents", "metadatas", "distances"]
)

# Mostrar resultados
print("\nResultados:")
for i, (doc, meta, dist) in enumerate(zip(resultados['documents'][0],
                                          resultados['metadatas'][0],
                                          resultados['distances'][0])):
    print(f"\n--- Resultado {i+1} (distancia: {dist:.4f}) ---")
    print(f"Documento: {meta['documento']}")
    print(f"Categoría: {meta['categoria']}")
    print(f"Fragmento:\n{doc[:500]}...")  # primeros 500 caracteres

/usr/local/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Consulta: ¿Cuáles son los requisitos para solicitar titulación?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Resultados:

--- Resultado 1 (distancia: 0.1907) ---
Documento: REGLAMENTO DOCENCIA ORDINARIA v3.5
Categoría: B
Fragmento:
Artículo 49°
. Requisitos específicos de ingreso: ejercicio y experiencia laboral: Son requisitos específicos de ingreso a la docencia ordinaria: el ejercicio o experiencia profesional, acumulativa, consecutiva o sucesiva, mínimo de tres (03) años como:

49.1. Profesional; o, 49.2. Docente universitario; o, 49.3. Pre-docente: Jefe de práctica / laboratorio de nivel universitario. La ayudantía de cátedra universitaria suma adicionalmente en la experiencia 49.4. Excepcional. Profesionales para la ...

--- Resultado 2 (distancia: 0.2124) ---
Documento: REGLAMENTO DE ESTUDIOS POSGRADO 2025
Categoría: B
Fragmento:
Artículo 269º
Artículo 269o Requisitos de la convalidación. Los requisitos para efectuar la convalidación en posgrado son los siguientes: 269.1. Solicitud dirigida al director de la EPG, presentada en secretaría general. 269.2. Certificado de estudios origina